# Preprocessing target data for use with Abil.py
Target (the variable you want to model) data needs to be put on the same coordinate grid as your predictor (environmental) data.
You'll need to load your observations, do any additional data cleaning (dropping certain methods, any out of range data), and then convert the coordinates to match the predictor data.

### Import required packages

In [1]:
import pandas as pd
import numpy as np
import xarray as xr

### Define your observation data filename and variable of interest

In [2]:
filename = '/home/mv23682/Documents/Abil_tutorial/data/g_hux_obs.csv'
varname = 'Gephyrocapsa huxleyi HET'

### Load data
I'm using usecols here to specifically extract the columns I need, as this is a large spreadsheet and contains a lot of extra information that I'd just drop later

In [3]:
d_raw = pd.read_csv(filename,usecols=['Latitude','Longitude','Depth','Month','Gephyrocapsa huxleyi HET'])

In [4]:
# Rename columns to match predictor data
d = d_raw.rename(columns={
    "Latitude":"lat", 
    "Longitude":"lon", 
    "Depth":"depth", 
    "Month":"time"})


# Subset dataframe to only necessary variable
d = d[['lat','lon','depth','time','Gephyrocapsa huxleyi HET']]

# Drop any rows with missing values
d.dropna(subset=["Gephyrocapsa huxleyi HET"], inplace=True)
d.set_index(["lat", "lon", "depth", "time"], inplace=True)


### Load environmental predictor data

In [5]:
# Load environment data
ds_env = xr.open_dataset('/home/mv23682/Documents/Abil_tutorial/data/env_data_pac.nc')
df_env = ds_env.to_dataframe().reset_index()
env_cols = ["temperature","no3","o2","PAR"]
df_env = df_env[env_cols + ["lat","lon","depth","time"]].set_index(["lat","lon","depth","time"])
print(f"Environment rows: {len(df_env)}")

# Filter only rows whose environmental data exists AND is fully non-NaN
valid_env_idx = df_env.dropna().index
d = d.loc[d.index.intersection(valid_env_idx)]
print(f"Rows after dropping samples without complete environmental data: {len(d)}")

Environment rows: 70680
Rows after dropping samples without complete environmental data: 19


### Use environmental predictor data domain to generate pseudo zeros

In [6]:
n_obs = d[varname].notna().sum()

# Get all possible grid cells from df_env
env_idx = df_env.index

# Get occupied grid cells from d
d_idx = d.index

# Find grid cells NOT in d
available_idx = env_idx.difference(d_idx).to_frame(index=False)

# randomly sample rows
sampled_idx = available_idx.sample(n=n_obs, replace=True).reset_index(drop=True)

# create zeros dataframe
zeros_df = sampled_idx.copy()
zeros_df[varname] = 0  # or whatever column name you want

# set multi-index again if needed
zeros_df = zeros_df.set_index(['lat', 'lon', 'depth', 'time'])

### Merge observations, pseudo zeros, and environmental predictors

In [7]:
# Concatenate original data and zeros
d = pd.concat([d, zeros_df], ignore_index=False)
print(f"Rows after adding pseudo-zeros: {len(d)}")

Rows after adding pseudo-zeros: 38


In [8]:
# Merge with environment data
d = d.join(df_env, how="inner").reset_index()
print(f"Rows after merging with environment: {len(d)}")

# Save combined CSV
outfile = "/home/mv23682/Documents/Abil_tutorial/data/training_g_hux_pac.csv"
d.to_csv(outfile, index=False)

print("\n=== Finished ===")
print("Saved combined CSV to:")
print(outfile)

Rows after merging with environment: 38

=== Finished ===
Saved combined CSV to:
/home/mv23682/Documents/Abil_tutorial/data/training_g_hux_pac.csv
